<a href="https://colab.research.google.com/github/SiyumiJayawardhane/sinhala-wsd/blob/textbook_docai_extraction/Textbook_docai_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Textbook → Document AI OCR → target-word sentences (v2)
**PDF → 15-page chunks → Document AI OCR → pages (visual order + confidence) → cleaning → sentence units with page numbers → target-word extraction**. The extraction rules and output columns are the same as the corpus notebook.

### 1. Install

In [2]:
!pip install -q --upgrade google-cloud-documentai pypdf openpyxl

### 2. Settings
* `START = 10`: the first 10 PDF pages are front matter and aren't OCR'd. `book_page = pdf_page - START`.
* `MIN_CONF = 0.85`: pages whose mean **token** confidence is below this are excluded and logged. The threshold comes from SiPaKosa (CHiPSAL 2026).which excluded pages below a mean OCR confidence of 0.85. SiPaKosa used character-level confidence; this notebook uses Document AI’s token confidence.
* `MIN_CHARS, MAX_CHARS, MIN_WORDS, MIN_SI = 20, 400, 4, 0.8`: the sentence filters recorded in the project plan (§4.9).
* outputs stay in `Research-WSD-New/textbooks/Ext_Final/sinhala_lit_g10_11_docai/`.

In [3]:
import os, re, io, json, glob, bisect, random, unicodedata, datetime
from collections import Counter, defaultdict
import pandas as pd
from google.colab import auth, drive
drive.mount("/content/drive")

BASE     = "/content/drive/MyDrive/Research-WSD-New"

PROJECT_ID   = "sinhala-wsd-ocr"
LOCATION     = "us"
PROCESSOR_ID = "7901bb5eb670c044"

BOOK = {
    "book_id":   "sinhala_lit_g10_11",
    "pdf":       f"{BASE}/textbooks/sinhala lit g-10-11.pdf",
    "title":     "සිංහල සාහිත්‍ය සංග්‍රහය (Grade 10–11)",
    "publisher": "Educational Publications Department",
    "grade":     "10-11", "year": "2020", "edition": "7th printing (first printed 2014)",
    "isbn":      "978-955-25-0381-8",
    "url":       "http://edupub.gov.lk/Administrator/Sinhala/10/sin%20lit%20g-10-11/sinhala%20lit%20g-10-11.pdf",
}
# skip front matter and OCR page limit 15
START, CHUNK = 10, 15
MIN_CONF = 0.85
MIN_CHARS, MAX_CHARS, MIN_WORDS, MIN_SI = 20, 400, 4, 0.8

OUT       = f'{BASE}/textbooks/Ext_Final/{BOOK["book_id"]}_docai'
CHUNK_DIR = f"{OUT}/chunks"      # 15-page PDFs made in block 4
RAW       = f"{OUT}/docai_raw"   # one saved OCR response per chunk
for d in (CHUNK_DIR, RAW): os.makedirs(d, exist_ok=True)
print("PDF:", "OK" if os.path.exists(BOOK["pdf"]) else "MISSING", "| out:", OUT)

Mounted at /content/drive
PDF: OK | out: /content/drive/MyDrive/Research-WSD-New/textbooks/Ext_Final/sinhala_lit_g10_11_docai


### 3. Log & Connect

In [4]:
auth.authenticate_user()
from google.api_core.client_options import ClientOptions
from google.cloud import documentai

client = documentai.DocumentProcessorServiceClient(client_options=ClientOptions(
    api_endpoint=f"{LOCATION}-documentai.googleapis.com",
    quota_project_id=PROJECT_ID))

proc = client.get_processor(name=client.processor_path(PROJECT_ID, LOCATION, PROCESSOR_ID))
assert proc.type_ == "OCR_PROCESSOR", f"wrong processor type: {proc.type_}"
assert proc.state.name == "ENABLED", f"processor is {proc.state.name}"

NAME = proc.default_processor_version
VERSION_USED = NAME.split("/")[-1]
ver = client.get_processor_version(name=NAME)
print("processor:", proc.display_name, "| type:", proc.type_, "| version:", VERSION_USED, f"({ver.display_name})")

processor: sinhala-ocr | type: OCR_PROCESSOR | version: pretrained-ocr-v2.1-2024-08-07 (Google Stable)


### 4. Split into 15-page chunks and OCR them
* The PDF is cut into 15-page chunk
* language_hints=["si"] tells the OCR model to expect Sinhala.
* enable_native_pdf_parsing=False: the PDF’s embedded text is legacy FM-Abhaya font (Latin keystrokes), so the OCR must read the page images instead.
* Every request goes to the pinned processor version.
* Responses are saved in docai_raw/

In [5]:
from pypdf import PdfReader, PdfWriter
from tqdm.auto import tqdm

# 1) cut the book into 15-page chunk PDFs (front matter skipped); existing chunks are reused
reader = PdfReader(BOOK["pdf"]); n = len(reader.pages)
chunks = []
for s in range(START, n, CHUNK):
    e = min(s + CHUNK, n); path = f"{CHUNK_DIR}/p{s + 1:03d}-{e:03d}.pdf"
    if not os.path.exists(path):
        w = PdfWriter()
        for i in range(s, e): w.add_page(reader.pages[i])
        w.write(path)
    chunks.append(path)
print("PDF pages:", n, "| chunks:", len(chunks))

# 2) OCR settings: Sinhala hint; read the page images, not the FM-font text layer
OCR_OPTIONS = documentai.ProcessOptions(ocr_config=documentai.OcrConfig(
    hints=documentai.OcrConfig.Hints(language_hints=["si"]), enable_native_pdf_parsing=False))

# 3) OCR each chunk once (pinned version) and save the full response as JSON
errors = []
for path in tqdm(chunks):
    out = f"{RAW}/{os.path.basename(path)[:-4]}.json"
    if os.path.exists(out):
        continue
    try:
        res = client.process_document(request=documentai.ProcessRequest(
            name=NAME, process_options=OCR_OPTIONS,
            raw_document=documentai.RawDocument(content=open(path, "rb").read(), mime_type="application/pdf")))
        expected = len(PdfReader(path).pages)
        assert len(res.document.pages) == expected, f"got {len(res.document.pages)} of {expected} pages"
        with open(out + ".tmp", "w", encoding="utf-8") as f:
            f.write(documentai.Document.to_json(res.document))
        os.replace(out + ".tmp", out)
    except Exception as ex:
        errors.append((os.path.basename(path), repr(ex)[:200]))
print("done:", len(glob.glob(f"{RAW}/p*.json")), "/", len(chunks), "| errors:", errors)

PDF pages: 140 | chunks: 9


  0%|          | 0/9 [00:00<?, ?it/s]

done: 9 / 9 | errors: []


5. Pages in visual order, with confidence

Reads every saved OCR response in docai_raw/ and builds one record per PDF page:

* text: the page rebuilt line by line in visual order. Lines are sorted top to bottom; pieces at the same height (for example a verse line and its rhyme word) are joined left to right.
* ocr_conf: the mean of the page’s token confidences (OCR quality). Used for the MIN_CONF = 0.85 page filter and reported.

In [6]:
def page_rows(page, full_text):
    """Rebuild the page line by line in VISUAL order using each line's position.
       Pieces at the same height (e.g. a verse line + its rhyme word) are joined into one row."""
    W, H = page.dimension.width or 1, page.dimension.height or 1
    items = []
    for ln in page.lines:
        v = ln.layout.bounding_poly.normalized_vertices
        if v:
            xs, ys = [p.x for p in v], [p.y for p in v]
        else:                                                    # only pixel coordinates: convert to 0-1
            v = ln.layout.bounding_poly.vertices
            xs, ys = [p.x / W for p in v], [p.y / H for p in v]
        t = "".join(full_text[int(s.start_index or 0):int(s.end_index)]
                    for s in ln.layout.text_anchor.text_segments).strip()
        if t and xs:
            items.append((min(ys), max(ys), min(xs), t))
    items.sort(key=lambda it: (it[0] + it[1]) / 2)               # top → bottom
    rows = []
    for y0, y1, x, t in items:
        c = (y0 + y1) / 2
        if rows and abs(c - rows[-1]["c"]) < rows[-1]["h"] * 0.5:   # same height → same row
            rows[-1]["parts"].append((x, t))
        else:
            rows.append({"c": c, "h": y1 - y0, "parts": [(x, t)]})
    return "\n".join(" ".join(t for x, t in sorted(r["parts"])) for r in rows)   # left → right

raw_files = sorted(glob.glob(f"{RAW}/p*.json"))
assert raw_files, "no OCR results in docai_raw/ - run block 4 first"
pages = []
for path in raw_files:
    first = int(re.match(r"p(\d+)-", os.path.basename(path)).group(1))     # p011-025.json -> 11
    doc = documentai.Document.from_json(open(path, encoding="utf-8").read(), ignore_unknown_fields=True)
    for k, page in enumerate(doc.pages):
        toks = [t.layout.confidence for t in page.tokens]
        pages.append({"pdf_page": first + k, "printed_page": first + k - START,
                      "ocr_conf": round(sum(toks) / len(toks), 4) if toks else 0.0,
                      "text": page_rows(page, doc.text)})
pages.sort(key=lambda p: p["pdf_page"])

missing = sorted(set(range(START + 1, n + 1)) - {p["pdf_page"] for p in pages})
assert not missing, f"pages missing from the OCR: {missing} - re-run block 4"
with open(f'{OUT}/{BOOK["book_id"]}_pages.jsonl', "w", encoding="utf-8") as f:
    f.writelines(json.dumps(p, ensure_ascii=False) + "\n" for p in pages)
print("pages:", len(pages), "| first:", pages[0]["pdf_page"], "| last:", pages[-1]["pdf_page"])

pages: 130 | first: 11 | last: 140
